<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🛠️ LangChain / LangGraph 工具类型大全

**一句话定位**:工具不止「执行动作的函数」—— 按「关心什么」可分 **5 大语义类型**,搞清类型 = 设计对 agent 的一半。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**心智模型(开局必须建立)**:

工具 = **LLM 的「词汇表」**,不是真的执行器。`bind_tools` 之后,LLM 学到的是「**我可以输出 JSON 参数,让某个工具被调用**」。**真正的执行** 是外部 Python 干的(`ToolNode` 或你手写 `tool.invoke(args)`)。

所以「工具」本质 = **schema(LLM 能看到)+ 一段可选的 Python 代码(LLM 看不到)**。

下面 5 大类型 = 这两部分的 **不同组合方式**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗂️ 5 大语义类型 — 按「工具关心什么」分

</div>

| # | 类型 | 工具核心关心 | 返回值的作用 | 经典例子 |
|---|------|--------------|--------------|----------|
| ① | **计算型** | 算结果(纯函数) | LLM 读了**继续推理** | `calculator` · `parse_date` · `extract_info` |
| ② | **副作用型** | 动作落地(I/O) | 只回「成功 / 失败」 | `send_email` · `write_file` · `http_post` |
| ③ | **查询型** | 取数据(读 I/O) | 数据**本身就是答案** | `tavily_search` · `ls` · `read_file` · `db_query` |
| ④ | **信号型** | 「**被调用**」这个事件本身 | 几乎为空,有也是占位 | `Done` · `Question` · `think_tool` |
| ⑤ | **状态变更型** | 改 LangGraph state | 返回 `Command(update, goto)` | `write_todos` · `task`(派单)· `edit_file` |

**判断套路**:看返回值 LLM **拿来干嘛**——继续算?知道动作完成?当数据?完全不看?改 state? → 5 类对号入座。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ① 计算型 + ② 副作用型

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># ① 计算型 ── 返回值是「真东西」,LLM 拿来继续推理
@tool
def calculator(a: float, b: float, op: str) -> float:
    """Compute a op b. op in (+, -, *, /)."""
    return {"+": a+b, "-": a-b, "*": a*b, "/": a/b}[op]
# 返回 42.0 → 下一轮 LLM 用 "42" 写答案

# ② 副作用型 ── 返回值只是「确认信号」
@tool
def send_email(to: str, subject: str, content: str) -> str:
    """Send an email."""
    try:
        smtp.send(to, subject, content)
        return <span style="background:#3d3414; color:#fde047; padding:0 2px;">"Email sent to " + to</span>      # ← 只是个 OK 标志
    except SMTPError as e:
        return <span style="background:#3d3414; color:#fde047; padding:0 2px;">f"FAILED: {e}"</span>            # ← 失败信息给 LLM 看,它会重试
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察:副作用型工具的错误信息「目标读者是 LLM」**

返回值写 `"OK"` / `"FAILED: connection timeout"` —— **不是给用户看的**,是给下一轮 LLM 看的。LLM 看到失败描述,会自己决定重试 / 换方式 / 放弃。所以 **错误信息要清晰自然语言**,不要塞 stacktrace。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ③ 查询型 + ④ 信号型(Done 揭秘)

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python"># ③ 查询型 ── 返回值「就是答案」,LLM 加工后回用户
@tool
def tavily_search(query: str) -> str:
    """Search the web."""
    return tavily_client.search(query)   # 数据本身

# ④ 信号型 ── 「被调用」这个动作本身就是消息;返回值不重要
from pydantic import BaseModel, Field

<span style="background:#3d3414; color:#fde047; padding:0 2px;">class Done(BaseModel):</span>
    """E-mail has been sent. Use this to signal task completion."""
    done: bool = Field(default=True, description="Always True.")

# 用法:
llm.bind_tools([send_email, schedule_meeting, <span style="background:#3d3414; color:#fde047; padding:0 2px;">Done</span>])
# LLM 想结束 → 输出 tool_call("Done", {done: True})
# conditional_edge 检测到名字是 "Done" → 跳到 END
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 关键洞察:`Done` 为什么是 class,不是 function?**

- `@tool` 装饰**函数** = 既给 LLM 看 schema,**也提供执行代码**
- 直接传 **Pydantic class** 给 `bind_tools` = **只**给 LLM 看 schema,**没有执行代码**(也不需要)

因为 **信号型工具的全部价值在「调用本身」**——具体怎么执行根本无所谓。`Done` 没有副作用要做,也没有数据要返回,所以**用 class 比用 function 更合适**(意图更明显)。

**同族成员**:
- `Question`(M4 HITL):LLM 调它 → 触发 interrupt 让用户回答
- `think_tool`(deep agents):LLM 调它 → ToolMessage 把 reflection 回写进 context(自言自语 / **recitation**)

**共同模式**:`tool_call` 作为「LLM 表达意图的词汇表」,不是「触发副作用的开关」。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⑤ 状态变更型 — LangGraph 特有

</div>

**关键能力**:工具能 **直接改 graph state**(不只回个字符串)。靠 `InjectedState` / `InjectedToolCallId` 拿到 state,然后返回 `Command(update=..., goto=...)`。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command
from langchain_core.messages import ToolMessage

@tool
def write_todos(
    todos: list[dict],
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">state: Annotated[dict, InjectedState]</span>,           # ← LLM 看不到
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">tool_call_id: Annotated[str, InjectedToolCallId]</span>,  # ← LLM 看不到
) -> Command:
    """Create or update the TODO list."""
    return <span style="background:#3d3414; color:#fde047; padding:0 2px;">Command(update={
        "todos": todos,                                       # 改 state.todos
        "messages": [ToolMessage("Updated todos", tool_call_id=tool_call_id)],
    })</span>
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 `Injected*` 系列 = 「LLM 看不到的隐藏参数」**

LLM 看到的 schema **只包含没被 Injected 标注的字段**(本例:`todos`)。`state` 和 `tool_call_id` 是 LangGraph 在 ToolNode 内部**自动注入**的——LLM 根本不知道它们存在,也就不会乱填。

**典型用途**:`task`(派子 agent,需要传父 state)· `edit_file`(需要拿到当前 files 字典)· `write_todos`(更新 todos 列表)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 4 种创建方式 — 按「代码长什么样」分

</div>

| 写法 | 一句话 | 何时用 | 对应语义类型 |
|------|--------|--------|---------------|
| `@tool` **函数** | 最常见,Python 函数→工具 | 90% 情况 | ① ② ③ ⑤ |
| **Pydantic class**(`BaseModel`)直接传给 `bind_tools` | 只要 schema,不要执行 | **信号型**专用 | ④ |
| `StructuredTool.from_function(...)` | 显式控制 name/description | 需要细调元信息 | 任意 |
| `class MyTool(BaseTool): ...` | 子类化,完全自定义 | 复杂工具,需要 `_run` / `_arun` 分离 | 任意 |

**另外两个常见来源**(不算「写法」,是「拿现成的」):

- 🔌 **LangChain 集成**:`TavilySearch()`、`GmailToolkit().get_tools()` —— 直接拿一组工具用
- 🌐 **MCP server**:用 `langchain-mcp-adapters` 把 MCP server 暴露的工具变成 LangChain 工具(跨语言、跨进程)

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**工具 = LLM 的词汇表 + 可选的 Python 执行**。

- 想让 LLM **用结果继续推理** → 计算型 ① / 查询型 ③
- 想让 LLM **触发外部动作** → 副作用型 ②(返回值是给 LLM 看的状态报告)
- 想让 LLM **表达意图 / 控制流** → 信号型 ④(`Done`、`Question`、`think_tool` — 用 **Pydantic class** 最优雅)
- 想让 LLM **直接改 graph state** → 状态变更型 ⑤(用 `InjectedState` + `Command`)

🎯 设计 agent 时,**先想清每个工具属于哪类**,再写 schema —— 类型对了,prompt 也好写,LLM 也用得稳。

</div>